# BFS Maze Solver — FPGA Test (PYNQ)

**Before running this notebook**, copy the following files to the same folder on your PYNQ board:
- `design_1_wrapper.bit`  — bitstream from Vivado  
- `design_1_wrapper.hwh`  — rename `design_1.hwh` → `design_1_wrapper.hwh` (PYNQ requires `.bit` and `.hwh` to share the same base name)

## Register Map (AXI-Lite base = `0x43C00000`)

| Offset | Name   | Description |
|--------|--------|-------------|
| 0x00   | CTRL   | Write `1` to bit[0] to start |
| 0x04   | STATUS | bit[0] = done |
| 0x08   | GOAL   | bits[3:0]=goal_x, bits[7:4]=goal_y |
| 0x0C   | CURR   | bits[3:0]=curr_x, bits[7:4]=curr_y |
| 0x10   | GRID0  | grid_free[31:0] |
| 0x14   | GRID1  | grid_free[63:32] |
| 0x18   | GRID2  | grid_free[95:64] |
| 0x1C   | GRID3  | grid_free[99:96] (bits[3:0] only) |
| 0x20   | OUT    | bits[6:0]=dist, bits[8:7]=next_dir, bit[9]=next_valid |

**`grid_free` encoding:** bit `i = y*10 + x` is **1 = free cell**, 0 = wall  
**`next_dir` encoding:** `00=N, 01=E, 10=S, 11=W`

## 1 — Load Overlay

In [ ]:
import os, time
from pynq import Overlay

# ── Edit this path if you put the files elsewhere ──────────────────────────
OVERLAY_PATH = "/home/xilinx/jupyter_notebooks/maze/design_1_wrapper.bit"
# ───────────────────────────────────────────────────────────────────────────

# PYNQ automatically loads the .hwh with the same base name.
# Make sure design_1_wrapper.hwh is in the same folder.
assert os.path.exists(OVERLAY_PATH), f"Bitstream not found: {OVERLAY_PATH}"
hwh_path = OVERLAY_PATH.replace(".bit", ".hwh")
assert os.path.exists(hwh_path), (
    f"HWH not found: {hwh_path}\n"
    "Rename design_1.hwh → design_1_wrapper.hwh and put it next to the .bit file."
)

ol = Overlay(OVERLAY_PATH)
print("Overlay loaded successfully")
print("Available IPs:", list(ol.ip_dict.keys()))

## 2 — Get the BFS IP handle

In [ ]:
# The IP is named bfs_solver_0 in the block design
ip = ol.bfs_solver_0
print("BFS IP base address:", hex(ip.mmio.base_addr))
print("BFS IP range       :", hex(ip.mmio.length))

## 3 — Helper Functions

In [ ]:
from typing import List, Tuple
from collections import deque
import random

W, H = 10, 10
DIR_NAMES = {0: 'N', 1: 'E', 2: 'S', 3: 'W'}
DIR_DELTA = {0: (0, -1), 1: (1, 0), 2: (0, 1), 3: (-1, 0)}

# ── Grid convention ────────────────────────────────────────────────────────
# grid[y][x] = 0  → free cell
# grid[y][x] = 1  → wall

def pack_grid(grid: List[List[int]]) -> List[int]:
    """
    Pack 10x10 grid into 4 x 32-bit words for AXI registers.
    Hardware grid_free: bit (y*10+x) = 1 means FREE, 0 means WALL.
    """
    bits = 0
    for y in range(H):
        for x in range(W):
            idx = y * W + x
            if grid[y][x] == 0:   # free cell → set bit
                bits |= (1 << idx)
    return [
        (bits >>  0) & 0xFFFF_FFFF,
        (bits >> 32) & 0xFFFF_FFFF,
        (bits >> 64) & 0xFFFF_FFFF,
        (bits >> 96) & 0xFFFF_FFFF,
    ]

def encode_coord(x: int, y: int) -> int:
    """Pack (x, y) into 8-bit register word: bits[3:0]=x, bits[7:4]=y."""
    return (x & 0xF) | ((y & 0xF) << 4)

def decode_out(out: int) -> Tuple[int, int, bool]:
    """Decode OUT register → (dist, next_dir, next_valid)."""
    dist      = out & 0x7F
    next_dir  = (out >> 7) & 0x3
    next_valid = bool((out >> 9) & 0x1)
    return dist, next_dir, next_valid

def print_maze(grid, path=None, start=(0,0), goal=(9,9)):
    """Pretty-print the maze. Path is a list of (x,y) tuples."""
    path_set = set(path) if path else set()
    for y in range(H):
        row = []
        for x in range(W):
            if (x, y) == start:
                row.append('S')
            elif (x, y) == goal:
                row.append('G')
            elif (x, y) in path_set:
                row.append('*')
            elif grid[y][x] == 1:
                row.append('#')
            else:
                row.append('.')
        print(' '.join(row))
    print()

# ── Solvability check (software, for maze generation) ──────────────────────
def is_solvable(grid, start=(0,0), goal=(9,9)):
    sx, sy = start
    gx, gy = goal
    if grid[sy][sx] == 1 or grid[gy][gx] == 1:
        return False
    q = deque([start])
    seen = {start}
    while q:
        x, y = q.popleft()
        if (x, y) == goal:
            return True
        for dx, dy in ((1,0),(-1,0),(0,1),(0,-1)):
            nx, ny = x+dx, y+dy
            if 0<=nx<W and 0<=ny<H and grid[ny][nx]==0 and (nx,ny) not in seen:
                seen.add((nx,ny))
                q.append((nx,ny))
    return False

def generate_maze(seed=42, wall_prob=0.28, max_tries=500):
    rng = random.Random(seed)
    for _ in range(max_tries):
        grid = [[0]*W for _ in range(H)]
        for y in range(H):
            for x in range(W):
                if (x,y) in ((0,0),(9,9)):
                    continue   # start and goal always free
                grid[y][x] = 1 if rng.random() < wall_prob else 0
        if is_solvable(grid):
            return grid
    raise RuntimeError("Could not generate solvable maze")

print("Helper functions defined.")

## 4 — Low-level AXI Driver

In [ ]:
# Register offsets
REG_CTRL   = 0x00
REG_STATUS = 0x04
REG_GOAL   = 0x08
REG_CURR   = 0x0C
REG_GRID0  = 0x10
REG_GRID1  = 0x14
REG_GRID2  = 0x18
REG_GRID3  = 0x1C
REG_OUT    = 0x20

def hw_load_maze(ip, grid, goal=(9,9)):
    """Write the maze grid and goal position to the hardware."""
    w0, w1, w2, w3 = pack_grid(grid)
    ip.write(REG_GRID0, w0)
    ip.write(REG_GRID1, w1)
    ip.write(REG_GRID2, w2)
    ip.write(REG_GRID3, w3)
    ip.write(REG_GOAL, encode_coord(*goal))
    print(f"Maze loaded. GRID words: {hex(w0)}, {hex(w1)}, {hex(w2)}, {hex(w3)}")
    print(f"Goal set to {goal}")

def hw_query(ip, curr_xy, timeout=0.5):
    """
    Write current position, pulse start, wait for done, read result.
    Returns (dist, next_dir, next_valid).
    """
    ip.write(REG_CURR, encode_coord(*curr_xy))
    ip.write(REG_CTRL, 0x1)   # start pulse

    t0 = time.time()
    while True:
        if ip.read(REG_STATUS) & 0x1:
            break
        if (time.time() - t0) > timeout:
            raise TimeoutError(f"HW solver timed out after {timeout}s")
    
    out = ip.read(REG_OUT)
    return decode_out(out)

print("AXI driver ready.")

## 5 — Sanity Check: Read-back Registers

Verify the AXI interface works before running the BFS.

In [ ]:
# Write known values and read back
TEST_GOAL = (3, 7)
ip.write(REG_GOAL, encode_coord(*TEST_GOAL))
rb = ip.read(REG_GOAL)

rb_x = rb & 0xF
rb_y = (rb >> 4) & 0xF
print(f"Wrote GOAL = {TEST_GOAL}, Read back: x={rb_x}, y={rb_y}")
assert (rb_x, rb_y) == TEST_GOAL, "Register read-back mismatch! Check AXI connection."
print("Register read-back OK")

## 6 — Test with a Simple Hand-crafted Maze

Start with a known maze so we can verify the answer independently.

In [ ]:
# Simple open maze (only right column is blocked except a single gap)
# 0 = free, 1 = wall
SIMPLE_MAZE = [
    [0,0,0,0,0,0,0,0,0,0],  # y=0
    [0,1,1,1,1,1,1,1,1,0],  # y=1
    [0,0,0,0,0,0,0,0,0,0],  # y=2
    [0,1,1,1,1,1,1,1,1,0],  # y=3
    [0,0,0,0,0,0,0,0,0,0],  # y=4
    [0,1,1,1,1,1,1,1,1,0],  # y=5
    [0,0,0,0,0,0,0,0,0,0],  # y=6
    [0,1,1,1,1,1,1,1,1,0],  # y=7
    [0,0,0,0,0,0,0,0,0,0],  # y=8
    [0,0,0,0,0,0,0,0,0,0],  # y=9
]

START = (0, 0)
GOAL  = (9, 9)

print("Simple maze (S=start, G=goal, #=wall, .=free):")
print_maze(SIMPLE_MAZE, start=START, goal=GOAL)
print(f"Solvable (software check): {is_solvable(SIMPLE_MAZE, START, GOAL)}")

## 7 — Run BFS on FPGA: Follow Path Step by Step

In [ ]:
def run_bfs_fpga(ip, grid, start=(0,0), goal=(9,9), max_steps=200, verbose=True):
    """
    Iteratively query the FPGA BFS core to navigate from start to goal.
    Each query: load current position → hardware runs BFS from goal → 
    hardware returns best next direction from current position.
    
    Returns list of (x,y) positions from start to goal (inclusive).
    """
    # Load maze + goal (only needed once per maze/goal combination)
    hw_load_maze(ip, grid, goal)
    print()

    curr = start
    path = [curr]
    visited = {curr}

    for step in range(max_steps):
        if curr == goal:
            print(f"Reached goal in {step} steps!")
            break

        dist, ndir, nvalid = hw_query(ip, curr)

        if verbose:
            dir_name = DIR_NAMES.get(ndir, '?')
            print(f"  Step {step+1:3d}: pos={curr}  dist_to_goal={dist}  "
                  f"next_dir={dir_name}  valid={nvalid}")

        if not nvalid:
            print("Hardware says no valid next move — no path exists or stuck.")
            break

        dx, dy = DIR_DELTA[ndir]
        nx, ny = curr[0] + dx, curr[1] + dy

        # Safety: check bounds and wall
        if not (0 <= nx < W and 0 <= ny < H):
            print(f"ERROR: HW directed out of bounds to ({nx},{ny})")
            break
        if grid[ny][nx] == 1:
            print(f"ERROR: HW directed into a wall at ({nx},{ny})")
            break
        if (nx, ny) in visited:
            print(f"WARNING: Cycle detected at ({nx},{ny}) — halting")
            break

        curr = (nx, ny)
        path.append(curr)
        visited.add(curr)
    else:
        print(f"Reached max_steps={max_steps} without finding goal.")

    return path


print("Running BFS on FPGA (simple maze)...")
path = run_bfs_fpga(ip, SIMPLE_MAZE, start=START, goal=GOAL)

print(f"\nPath length: {len(path)} steps")
print(f"Path: {path}")
print("\nMaze with path (* = visited):")
print_maze(SIMPLE_MAZE, path=path[1:-1], start=START, goal=GOAL)

## 8 — Test with a Random Maze

In [ ]:
SEED = 42
rand_maze = generate_maze(seed=SEED, wall_prob=0.28)

print(f"Random maze (seed={SEED}):")
print_maze(rand_maze, start=START, goal=GOAL)
print(f"Software solvability check: {is_solvable(rand_maze, START, GOAL)}")

In [ ]:
print("Running BFS on FPGA (random maze)...")
path_rand = run_bfs_fpga(ip, rand_maze, start=START, goal=GOAL, verbose=True)

print(f"\nPath length : {len(path_rand)} steps")
print(f"Path        : {path_rand}")
print("\nMaze with FPGA-found path:")
print_maze(rand_maze, path=path_rand[1:-1], start=START, goal=GOAL)

## 9 — Verify: Compare FPGA Path vs Software BFS

In [ ]:
def software_bfs(grid, start=(0,0), goal=(9,9)):
    """Returns shortest path as list of (x,y), or None if unreachable."""
    q = deque([(start, [start])])
    seen = {start}
    while q:
        (x,y), path = q.popleft()
        if (x,y) == goal:
            return path
        for dx,dy in ((1,0),(-1,0),(0,1),(0,-1)):
            nx,ny = x+dx, y+dy
            if 0<=nx<W and 0<=ny<H and grid[ny][nx]==0 and (nx,ny) not in seen:
                seen.add((nx,ny))
                q.append(((nx,ny), path+[(nx,ny)]))
    return None

sw_path = software_bfs(rand_maze, START, GOAL)
hw_path = path_rand

print(f"Software BFS path length : {len(sw_path) if sw_path else 'N/A'}")
print(f"FPGA BFS path length     : {len(hw_path)}")

if sw_path and hw_path and hw_path[-1] == GOAL:
    if len(hw_path) == len(sw_path):
        print("\nPASS: FPGA found optimal (shortest) path")
    else:
        print(f"\nINFO: FPGA path is {len(hw_path)-len(sw_path)} steps longer "
              f"than optimal (both reach the goal)")
else:
    print("\nFAIL or unreachable")

## 10 — Benchmark: Measure BFS Latency per Query

In [ ]:
import time

hw_load_maze(ip, rand_maze, goal=GOAL)

N = 50
test_pos = (0, 0)

t_start = time.perf_counter()
for _ in range(N):
    hw_query(ip, test_pos)
t_end = time.perf_counter()

avg_ms = (t_end - t_start) / N * 1000
print(f"Average HW BFS query time: {avg_ms:.3f} ms  ({N} runs)")
print(f"(Includes AXI write + BFS compute + AXI read overhead)")

## 11 — Multi-Maze Stress Test

In [ ]:
NUM_MAZES = 20
passed = 0
failed = 0
total_path_diff = 0

for seed in range(NUM_MAZES):
    try:
        maze = generate_maze(seed=seed, wall_prob=0.28)
    except RuntimeError:
        continue

    sw_p = software_bfs(maze, START, GOAL)
    hw_p = run_bfs_fpga(ip, maze, start=START, goal=GOAL, verbose=False)

    if hw_p and hw_p[-1] == GOAL:
        diff = len(hw_p) - (len(sw_p) if sw_p else 0)
        total_path_diff += diff
        status = "PASS" if diff == 0 else f"suboptimal+{diff}"
        passed += 1
    else:
        status = "FAIL"
        failed += 1

    print(f"  seed={seed:3d}  sw_len={len(sw_p) if sw_p else 'X':3}  "
          f"hw_len={len(hw_p):3d}  {status}")

print(f"\nResults: {passed} passed, {failed} failed out of {NUM_MAZES} mazes")
if passed:
    print(f"Average path length overhead vs optimal: {total_path_diff/passed:.2f} steps")

## Troubleshooting

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| `IP not found` error | Wrong IP name | Check `ol.ip_dict.keys()` output |
| `TimeoutError` | HW never asserts done | Check reset wiring; try power cycling |
| Register readback wrong | AXI address off | Confirm base addr in HWH = `0x43C00000` |
| FPGA path goes into wall | Grid bit packing | Grid must encode **1=free**, not 1=wall |
| Path longer than optimal | Normal | Hardware returns ONE next step per query; ties broken by N→E→S→W priority |